In [1]:
# ==============================
# INSTALL (run once if needed)
# ==============================
# !pip install panel hvplot pandas plotly numpy

import pandas as pd
import numpy as np
import panel as pn
import hvplot.pandas
import plotly.express as px

pn.extension('plotly')

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("C:/Users/Varnika Sharma/Desktop/Crime Analysis/Crime Analysis/01_Data/Final_Dataset/cleaned_crime_data.csv")   # <-- UPDATE PATH

# ==============================
# FILTERS
# ==============================
year_slider = pn.widgets.IntRangeSlider(
    name="Year Range",
    start=int(df["Year"].min()),
    end=int(df["Year"].max()),
    value=(int(df["Year"].min()), int(df["Year"].max()))
)

state_select = pn.widgets.MultiChoice(
    name="Select States",
    options=sorted(df["State"].unique().tolist()),
    value=sorted(df["State"].unique().tolist())[:10]
)

dataset_select = pn.widgets.MultiChoice(
    name="Crime Category",
    options=sorted(df["Dataset"].unique().tolist()),
    value=sorted(df["Dataset"].unique().tolist())
)

# ==============================
# FILTER FUNCTION
# ==============================
def get_data():
    return df[
        (df["Year"].between(*year_slider.value)) &
        (df["State"].isin(state_select.value)) &
        (df["Dataset"].isin(dataset_select.value))
    ]

# ==============================
# 1. SCATTER
# ==============================
def scatter_plot():
    data = get_data()

    pivot = (
        data.groupby(["State", "Dataset"])["Value"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )

    return pivot.hvplot.scatter(
        x="Women",
        y="Cyber",
        size="Children",
        hover_cols=["State"],
        height=350,
        title="Women vs Cyber Crimes"
    )

# ==============================
# 2. STACKED BAR
# ==============================
def stacked_bar():
    data = get_data()

    grouped = (
        data.groupby(["Year", "Dataset"])["Value"]
        .sum()
        .unstack()
        .fillna(0)
    )

    return grouped.hvplot.bar(
        stacked=True,
        height=350,
        title="Crime Distribution Over Years"
    )

# ==============================
# 3. PIE (COLORED)
# ==============================
def pie_chart():
    data = get_data()

    pie_df = (
        data.groupby("Dataset")["Value"]
        .sum()
        .reset_index()
        .sort_values("Value", ascending=False)
        .head(6)
    )

    fig = px.pie(
        pie_df,
        names="Dataset",
        values="Value",
        hole=0.5,
        title="Crime Share Distribution",
        color="Dataset",
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    return pn.pane.Plotly(fig, height=400)

# ==============================
# 4. STATE BAR (COLORED)
# ==============================
def state_bar():
    data = get_data()

    state_df = (
        data.groupby(["State", "Dataset"])["Value"]
        .sum()
        .reset_index()
    )

    fig = px.bar(
        state_df,
        x="State",
        y="Value",
        color="Dataset",
        title="State-wise Crime Distribution",
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    return pn.pane.Plotly(fig, height=400)

# ==============================
# 5. TREND LINE (COLORED)
# ==============================
def trend_line():
    data = get_data()

    trend_df = (
        data.groupby("Year")["Value"]
        .sum()
        .reset_index()
    )

    fig = px.line(
        trend_df,
        x="Year",
        y="Value",
        markers=True,
        title="Year-wise Crime Trend",
        color_discrete_sequence=["#2a9d8f"]   # clean teal
    )

    return pn.pane.Plotly(fig, height=400)

# ==============================
# 6. GROWTH RATE (COLORED)
# ==============================
def growth_chart():
    data = get_data()

    df_year = (
        data.groupby("Year")["Value"]
        .sum()
        .reset_index()
    )

    df_year["Growth"] = df_year["Value"].pct_change() * 100
    df_year = df_year.dropna()

    fig = px.bar(
        df_year,
        x="Year",
        y="Growth",
        title="Year-wise Crime Growth Rate (%)",
        color="Growth",
        color_continuous_scale="RdYlGn"
    )

    return pn.pane.Plotly(fig, height=400)

# ==============================
# BIND
# ==============================
scatter_p = pn.bind(scatter_plot)
bar_p = pn.bind(stacked_bar)
pie_p = pn.bind(pie_chart)
state_p = pn.bind(state_bar)
trend_p = pn.bind(trend_line)
growth_p = pn.bind(growth_chart)

# ==============================
# DASHBOARD
# ==============================
dashboard = pn.template.FastListTemplate(
    title="Crime Analysis Dashboard (India)",

    sidebar=[
        pn.pane.Markdown("## Filters"),
        year_slider,
        state_select,
        dataset_select
    ],

    main=[
        pn.pane.Markdown("## Overview"),

        pn.Row(scatter_p, bar_p),

        pn.Row(pie_p, state_p),

        pn.Row(trend_p, growth_p)
    ]
)

# ==============================
# RUN
# ==============================
dashboard.show()

Launching server at http://localhost:63772
